In [1]:
from pathlib import Path

import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from nre.preprocess import preprocess_df
from nre.network_connectivity import ConnectivityUnit, ton_iot_conn_param_specs
from nre.analyze_nf_dataset import nre_classification, flow_based_classification
from nre.classification_tools import plot_roc_curves
from nre.validation_tools import validate_method

from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB


# CIC-IDS-2017

In [56]:

# CIC-IDS-2017
file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Tuesday-WorkingHours.pcap_ISCX.csv'  # 'Monday-WorkingHours.pcap_ISCX.csv' #  Wednesday-workingHours.pcap_ISCX.csv
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Friday-WorkingHours-Morning.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'

# NF-ToN-IoT
#file_addr = r'../../NF-ToN-IoT/NF-ToN-IoT-v3-small.csv'
file_addr = r'../../NF-ToN-IoT/NF-ToN-IoT-v3.csv'

file_addr = Path(file_addr)

# Definitions
date_col = 'FLOW_START_MILLISECONDS' # 'FLOW_START_MILLISECONDS' # ' Timestamp'
label_col = 'Label' # 'Label' # ' Label'
benign_label = '0'  # '0' # 'BENIGN'
src_id_col = 'IPV4_SRC_ADDR' # ' Source IP'
dst_id_col = 'IPV4_DST_ADDR' # ' Total Backward Packets'
src_feature_col = 'OUT_PKTS' # ' Total Fwd Packets'
dst_feature_col = 'IN_PKTS' # ' Total Backward Packets'
conn_param = 'NPS'
conn_param_specs = ton_iot_conn_param_specs
feat_cols = ('OUT_PKTS', 'IN_PKTS')

In [57]:
#df_cic = pd.read_csv(file_addr, header=0, encoding='cp1252')
df_iot = pd.read_csv(file_addr, sep=',', encoding='utf-8')

#df = preprocess_df(df_cic, date_col=' Timestamp')
df_whole = preprocess_df(df_iot, date_col=date_col, ds_type='ton-iot')

print(df_whole.shape)

(27520260, 55)


In [58]:
df_whole.head()

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,2019-04-23 14:09:50.618,1556028590618,192.168.1.32,41315,192.168.1.169,10010,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
1,2019-04-23 14:09:50.618,1556028590621,192.168.1.32,41315,192.168.1.194,49400,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
2,2019-04-23 14:09:50.618,1556028590621,192.168.1.32,41315,192.168.1.193,8008,6,161,44,1,...,0,0,0,0,0,0,0,0,1,scanning
3,2019-04-23 14:09:50.624,1556028590628,192.168.1.32,41315,192.168.1.194,85,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
4,2019-04-23 14:09:50.624,1556028590624,192.168.1.32,41317,192.168.1.180,1056,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign


# Train - Test Split
Use the same split (seed) in experiment section

In [88]:
seed = 138
test_size = 0.33

ind_co = int(df_whole.shape[0] * (1 - test_size))
df_train_val, df_test = df_whole.iloc[:ind_co, :], df_whole.iloc[ind_co:, :]

In [89]:
df_train_val.shape, df_test.shape

((18438574, 55), (9081686, 55))

In [90]:
np.unique(df_train_val[label_col].values, return_counts= True)

(array(['0', '1'], dtype=object), array([13221986,  5216588]))

In [91]:
type(df_train_val[label_col][0])

str

# Network Scope 

## CIC-IDS-2017

In [6]:
with open(r'saves\victim_net.pickle', 'rb') as handle:
    entity_names = pickle.load(handle) 
len(entity_names)

13

In [7]:
with open(r'saves\partitioned_nodes_141.pickle', 'rb') as handle: #105
    subnet_names = pickle.load(handle) 
len(subnet_names)

141

## NF-ToN-IoT

In [87]:
with open(r'saves/ton_iot_small_freq_1000.pickle', 'rb') as handle: #105
    subnet_names = pickle.load(handle) 
entity_names = subnet_names.copy()
len(subnet_names)

15

In [63]:
with open(r'saves/partitioned_nodes_iot_261.pickle', 'rb') as handle: #105
    subnet_names = pickle.load(handle) 
entity_names = subnet_names.copy()
len(subnet_names)

261

# Validation

In [64]:
import importlib
import nre.validation_tools

importlib.reload(nre.validation_tools)
from nre.validation_tools import *

In [65]:
import importlib
import nre.kalman_network_tools

importlib.reload(nre.kalman_network_tools)
from nre.kalman_network_tools import *

In [98]:
import importlib
import nre.analyze_nf_dataset

importlib.reload(nre.analyze_nf_dataset)
from nre.analyze_nf_dataset import *

In [92]:
idx = df_train_val[src_id_col].isin(subnet_names) & df_train_val[dst_id_col].isin(subnet_names)
df_train_val = df_train_val[idx].copy()
df_train_val.shape

(11620501, 55)

In [93]:
df_train_val

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,2019-04-23 14:09:50.618,1556028590618,192.168.1.32,41315,192.168.1.169,10010,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
1,2019-04-23 14:09:50.618,1556028590621,192.168.1.32,41315,192.168.1.194,49400,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
2,2019-04-23 14:09:50.618,1556028590621,192.168.1.32,41315,192.168.1.193,8008,6,161,44,1,...,0,0,0,0,0,0,0,0,1,scanning
3,2019-04-23 14:09:50.624,1556028590628,192.168.1.32,41315,192.168.1.194,85,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
4,2019-04-23 14:09:50.624,1556028590624,192.168.1.32,41317,192.168.1.180,1056,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18438437,2019-04-26 02:14:42.366,1556244882366,192.168.1.194,80,192.168.1.31,58536,6,7,60,1,...,0,0,0,0,0,0,0,0,0,Benign
18438439,2019-04-26 02:14:42.366,1556244882366,192.168.1.194,80,192.168.1.31,58128,6,7,60,1,...,0,0,0,0,0,0,0,0,0,Benign
18438476,2019-04-26 02:14:42.479,1556244882481,192.168.1.30,56745,192.168.1.1,53,17,5,67,1,...,0,0,0,0,0,0,0,0,0,Benign
18438501,2019-04-26 02:14:42.492,1556244882496,192.168.1.30,42058,192.168.1.1,53,17,5,71,1,...,0,0,0,0,0,0,0,0,1,ddos


In [94]:
benign_label

'0'

In [99]:

def model_nre(df_train, ml_models, **kwargs):
    return nre_classification(df_train, ml_models, entity_names=subnet_names, standardize=True, **kwargs)  # entity_names=subnet_names

def model_fb(df_train, ml_models, **kwargs):
    return flow_based_classification(df_train, ml_models, entity_names=None, standardize=True, **kwargs)  # entity_names=entity_names

val_size = 0.25
n_train_val = df_train_val.shape[0]
n_val = int(n_train_val * val_size)
n_train = n_train_val - n_val

df_train = df_train_val.iloc[n_train:n_train_val, :]
df_val = df_train_val.iloc[:n_train, :]


# CIC-IDS-2017
"""
param_list = [{'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 90, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 45, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 360, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 720, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 0.6, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 0.3, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 2.4, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 4.8, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.2, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.8, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 3},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 10}
              ]
"""

# ToN-IoT
okwargs = {'date_col':date_col, 'src_id_col': src_id_col, 'dst_id_col': dst_id_col, 'conn_param':conn_param, 'window_type':'conn',
          'label_col':label_col, 'conn_param_specs':ton_iot_conn_param_specs, 'benign_label': benign_label, 'seed':345, 'verbose':False,
          'labelling_opt': 'attacks first'}
param_list = [{'n_graph': 10_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              #{'n_graph': 8_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              #{'n_graph': 5_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              #{'n_graph': 2_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              #{'n_graph': 5_000, 'conn_size':50, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              #{'n_graph': 5_000, 'conn_size':25, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              #{'n_graph': 5_000, 'conn_size':200, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              #{'n_graph': 5_000, 'conn_size':400, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              #{'n_graph': 5_000, 'conn_size':100, 'forget_factor': 0.2, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              #{'n_graph': 5_000, 'conn_size':100, 'forget_factor': 0.8, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              #{'n_graph': 5_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 3, **okwargs},
              #{'n_graph': 5_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 10, **okwargs}
              ]


okwargs = {'date_col':date_col, 'src_id_col': src_id_col, 'dst_id_col': dst_id_col, 'conn_param':conn_param,
           'window_type':'time',
          'label_col':label_col, 'conn_param_specs':ton_iot_conn_param_specs, 'benign_label': benign_label,
           'seed':345, 'verbose':False, 'labelling_opt': 'attacks first'}

param_list = [{'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'t_graph': 90, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'t_graph': 45, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'t_graph': 360, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'t_graph': 720, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'t_graph': 180, 'sync_window_size': 0.6, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'t_graph': 180, 'sync_window_size': 2.4, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'t_graph': 180, 'sync_window_size': 4.8, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.2, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.8, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 3, **okwargs},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 10, **okwargs}
              ]


param_list_fb = [{**params, 'feat_cols': feat_cols, 'labelling_opt':okwargs['labelling_opt']} for params in param_list]

auc_scores_nre = validate_method(df_train, df_val, model_nre, param_list)
auc_scores_fb = validate_method(df_train, df_val, model_fb, param_list_fb)

 25%|████████████████████████████████████▎                                                                                                            | 3/12 [3:13:34<10:55:55, 4372.87s/it]/home/ab126/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2394: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/ab126/.local/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1123: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(
/home/ab126/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2394: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/ab126/.local/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1123: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(
/home/ab126/.local/lib/p

 50%|█████████████████████████████████████████████████████████████████████████                                                                         | 6/12 [3:43:58<3:02:48, 1828.02s/it]/media/large_ssd/ab126/PycharmProjects/NRE/nre/analyze_nf_dataset.py:331: UserWarning: Reported results are not fully representative since thresholding is not implemented for the flow_based_classification function directly. Use the roc_curves parameters to gauge the performance instead.
  warnings.warn('Reported results are not fully representative since thresholding is not implemented for the ' +
 58%|█████████████████████████████████████████████████████████████████████████████████████▏                                                            | 7/12 [4:10:14<2:25:28, 1745.69s/it]/media/large_ssd/ab126/PycharmProjects/NRE/nre/analyze_nf_dataset.py:331: UserWarning: Reported results are not fully representative since thresholding is not implemented for the flow_based_classification function directly.

In [100]:
auc_scores_nre

[({'t_graph': 180,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1,
   'date_col': 'FLOW_START_MILLISECONDS',
   'src_id_col': 'IPV4_SRC_ADDR',
   'dst_id_col': 'IPV4_DST_ADDR',
   'conn_param': 'NPS',
   'window_type': 'time',
   'label_col': 'Label',
   'conn_param_specs': {'Activation': {'method': 'activation'},
    'Bytes Sent': {'method': 'total',
     'src_feature_col': 'OUT_BYTES',
     'dst_feature_col': 'IN_BYTES'},
    'Bytes Received': {'method': 'total',
     'src_feature_col': 'IN_BYTES',
     'dst_feature_col': 'OUT_BYTES'},
    'Flow Duration': {'method': 'total',
     'src_feature_col': 'FLOW_DURATION_MILLISECONDS',
     'dst_feature_col': 'FLOW_DURATION_MILLISECONDS'},
    'NPS': {'method': 'total',
     'src_feature_col': 'OUT_PKTS',
     'dst_feature_col': 'IN_PKTS'},
    'NPR': {'method': 'total',
     'src_feature_col': 'IN_PKTS',
     'dst_feature_col': 'OUT_PKTS'},
    'Port Number': {'method': 'mode',
     'src_featu

In [101]:
auc_scores_fb

[({'t_graph': 180,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1,
   'date_col': 'FLOW_START_MILLISECONDS',
   'src_id_col': 'IPV4_SRC_ADDR',
   'dst_id_col': 'IPV4_DST_ADDR',
   'conn_param': 'NPS',
   'window_type': 'time',
   'label_col': 'Label',
   'conn_param_specs': {'Activation': {'method': 'activation'},
    'Bytes Sent': {'method': 'total',
     'src_feature_col': 'OUT_BYTES',
     'dst_feature_col': 'IN_BYTES'},
    'Bytes Received': {'method': 'total',
     'src_feature_col': 'IN_BYTES',
     'dst_feature_col': 'OUT_BYTES'},
    'Flow Duration': {'method': 'total',
     'src_feature_col': 'FLOW_DURATION_MILLISECONDS',
     'dst_feature_col': 'FLOW_DURATION_MILLISECONDS'},
    'NPS': {'method': 'total',
     'src_feature_col': 'OUT_PKTS',
     'dst_feature_col': 'IN_PKTS'},
    'NPR': {'method': 'total',
     'src_feature_col': 'IN_PKTS',
     'dst_feature_col': 'OUT_PKTS'},
    'Port Number': {'method': 'mode',
     'src_featu

In [ ]:
np.where(df_train[label_col] == 1)

In [12]:
res = (auc_scores_nre, auc_scores_fb)
with open(r'saves\validation_curves_141.pickle', 'wb') as handle:
    pickle.dump(res, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [102]:
res = (auc_scores_nre, auc_scores_fb)
with open(r'saves\validation_curves_iot_15.pickle', 'wb') as handle:
    pickle.dump(res, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [29]:
with open(r'saves\validation_curves_iot_15.pickle', 'rb') as handle:
    auc_scores_nre, auc_scores_fb = pickle.load(handle) 

In [30]:
auc_scores_nre

[({'n_graph': 10000,
   'conn_size': 100,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1,
   'date_col': 'FLOW_START_MILLISECONDS',
   'src_id_col': 'IPV4_SRC_ADDR',
   'dst_id_col': 'IPV4_DST_ADDR',
   'conn_param': 'NPS',
   'window_type': 'conn',
   'label_col': 'Label',
   'conn_param_specs': {'Activation': {'method': 'activation'},
    'Bytes Sent': {'method': 'total',
     'src_feature_col': 'OUT_BYTES',
     'dst_feature_col': 'IN_BYTES'},
    'Bytes Received': {'method': 'total',
     'src_feature_col': 'IN_BYTES',
     'dst_feature_col': 'OUT_BYTES'},
    'Flow Duration': {'method': 'total',
     'src_feature_col': 'FLOW_DURATION_MILLISECONDS',
     'dst_feature_col': 'FLOW_DURATION_MILLISECONDS'},
    'NPS': {'method': 'total',
     'src_feature_col': 'OUT_PKTS',
     'dst_feature_col': 'IN_PKTS'},
    'NPR': {'method': 'total',
     'src_feature_col': 'IN_PKTS',
     'dst_feature_col': 'OUT_PKTS'},
    'Port Number': {'method': 'mode',
     'src_feature_co

In [31]:
auc_scores_fb

[({'n_graph': 10000,
   'conn_size': 100,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1,
   'date_col': 'FLOW_START_MILLISECONDS',
   'src_id_col': 'IPV4_SRC_ADDR',
   'dst_id_col': 'IPV4_DST_ADDR',
   'conn_param': 'NPS',
   'window_type': 'conn',
   'label_col': 'Label',
   'conn_param_specs': {'Activation': {'method': 'activation'},
    'Bytes Sent': {'method': 'total',
     'src_feature_col': 'OUT_BYTES',
     'dst_feature_col': 'IN_BYTES'},
    'Bytes Received': {'method': 'total',
     'src_feature_col': 'IN_BYTES',
     'dst_feature_col': 'OUT_BYTES'},
    'Flow Duration': {'method': 'total',
     'src_feature_col': 'FLOW_DURATION_MILLISECONDS',
     'dst_feature_col': 'FLOW_DURATION_MILLISECONDS'},
    'NPS': {'method': 'total',
     'src_feature_col': 'OUT_PKTS',
     'dst_feature_col': 'IN_PKTS'},
    'NPR': {'method': 'total',
     'src_feature_col': 'IN_PKTS',
     'dst_feature_col': 'OUT_PKTS'},
    'Port Number': {'method': 'mode',
     'src_feature_co

In [18]:
param_list = [{'t_graph': 90, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1}]
auc_scores_nre = validate_model(df_train, df_val, model_nre, param_list)
auc_scores_nre

  0%|                                                                                                                          | 0/1 [00:00<?, ?it/s]C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:23<00:00, 83.35s/it]


[({'t_graph': 90,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.9541666666666667)]

In [19]:
param_list = [{'t_graph': 360, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1}]
auc_scores_fb = validate_model(df_train, df_val, model_fb, param_list)
auc_scores_fb

  0%|                                                                                                                          | 0/1 [00:00<?, ?it/s]C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:15<00:00, 15.56s/it]


[({'t_graph': 360,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.7727272727272727)]